In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [2]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [3]:
to_predict_mens = pd.read_csv("to_predict_mens.csv")

In [4]:
to_predict_mens.sort_values(by="t1_adj_margin", ascending=False)[["t1_TeamName", "Season"]]

,t1_TeamName,Season
1082,Gonzaga,2019
1052,Gonzaga,2019
1096,Gonzaga,2019
2474,Gonzaga,2019
831,Kentucky,2015
...,...,...
1699,MS Valley St,2008
2417,NC Central,2019
1399,UNC Asheville,2003
0,UNC Asheville,2003


### Evaluate Impact on First Round Stats Model

In [5]:
# Define the classifier and parameter grid
model = LogisticRegression(C=0.05)
pipeline = make_pipeline(StandardScaler(), model)
param_grid = {
    'logisticregression__C': [.005, 0.001, .05, 0.01, 0.1],
}

In [6]:
to_predict_mens_first_round = to_predict_mens[(to_predict_mens["GameRound"] == 1)
                                                    # & (to_predict_mens.final_odds.notnull())
                                                    ]

In [7]:
to_predict_mens_first_round_recent = to_predict_mens_first_round[to_predict_mens_first_round.Season >= 2009]

In [8]:
#to_predict_mens_first_round_recent[to_predict_mens_first_round_recent["t1_FTR_mean"].isnull()][["t1_TeamName", "Season"]]

In [9]:
baseline_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank']
eval_df = validation.run_evaluation_framework(to_predict_mens_first_round_recent, pipeline, baseline_features, param_grid, cv_start=2013)

In [10]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.172203,"(-0.1744695251929034, -0.1699373479893921)",0.169975


In [22]:
to_predict_mens_first_round_recent[to_predict_mens_first_round_recent["t1_top5_STL_cv"].isna()][["t1_TeamName", "t1_Seed", "Season"]]

,t1_TeamName,t1_Seed,Season
605,NC State,11,2012
789,NC State,8,2015
1317,NC State,11,2024
2049,NC State,8,2013
2096,NC State,12,2014
2363,NC State,9,2018
2633,NC State,11,2023


In [11]:
# baseline_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank',
#             't1_injured_players_value',	't2_injured_players_value',
            # 't1_Min%_stdev', 't2_Min%_stdev',
            # 't1_PRPG!_max',	't2_PRPG!_max',
            # 't1_USG_stdev','t2_USG_stdev',
            # 't1_TS_stdev', 't2_TS_stdev',
            # 't1_OR_mean', 't2_OR_mean',
            # 't1_AST_mean', 't2_AST_mean',
            # 't1_BLK_cv', 't2_BLK_cv',
            # 't1_STL_cv', 't2_STL_cv',
            # 't1_FTR_mean', 't2_FTR_mean'
            #]

baseline_features = [
                    't1_adj_margin', 't2_adj_margin', 
                     't1_final_rank', 't2_final_rank',
                    't1_OrdinalRank', 't2_OrdinalRank',
                    't1_BPM_mean', 't2_BPM_mean',
                    't1_TO_stdev','t2_TO_stdev',
                    't1_AST_mean', 't2_AST_mean',
                    't1_TS_gini', 't2_TS_gini',
                    't1_Min%_gini', 't2_Min%_gini',
                    't1_FTR_mean', 't2_FTR_mean',
                    # 't1_health_score', 't2_health_score',
                    't1_BLK_cv', 't2_BLK_cv',
                    # 't1_USG_gini', 't2_USG_gini'
                    ]

# LATEST
baseline_features =    ['t1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean',
    't1_top8_TO_stdev', 't2_top8_TO_stdev',
    't1_top5_PRPG!_median', 't2_top5_PRPG!_median',
    't1_top3_DR_median', 't2_top3_DR_median',
    't1_top5_STL_cv', 't2_top5_STL_cv',
    't1_top3_Min%_median', 't2_top3_Min%_median',
    't1_top8_TS_gini', 't2_top8_TS_gini',
    't1_top3_USG_gini', 't2_top3_USG_gini',
    't1_OrdinalRank', 't2_OrdinalRank',
    't1_adj_margin', 't2_adj_margin'] 


eval_df = validation.run_evaluation_framework(to_predict_mens_first_round_recent, pipeline, baseline_features, param_grid, cv_start=2013)

In [12]:
# 0.159601
# 0.159501
# 0.160371
# 0.159
# 0.159141
# 0.160895
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.1},-0.160819,"(-0.1632760953555247, -0.15836182658384831)",0.15843


In [75]:
odds_features = ['final_odds']


first_round_df = to_predict_mens[(to_predict_mens.GameRound == 1)].copy()


first_round_df = first_round_df[~first_round_df.final_odds.isna()]

In [92]:
eval_df = validation.run_evaluation_framework(first_round_df, pipeline, odds_features, param_grid, cv_start=2013)

[[-1.59826703]]
[[-1.5784245]]
[[-1.60523263]]
[[-1.64188448]]
[[-1.63033733]]
[[-1.70001017]]
[[-1.678187]]
[[-1.68835753]]
[[-1.66308601]]
[[-1.64665667]]
[[-1.62507845]]


In [93]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.1},-0.16766,"(-0.1694484811181598, -0.16587151531125582)",0.172134


The new injury feature improves the statistics model.

Odds is worse though, which doesn't seem right 

Best New Injury Features: -0.171112, 0.168546

Baseline Odds Model (using all data, but eval rolling season on same years): -0.167682, 0.172134

Baseline Stats Model: -0.172153, 0.169975

Baseline Stats Model using all data _, 0.171663

Latest Update with more stat column: -0.163012, 0.159

Caveat - of course there's some leakage here since we identified missing players by whether or not they played, which we wouldn't know for sure until the games


In [ ]:
# Next Steps -> 
# Try to see if weighting the BPM by minutes improves the quality of the metric 
# Try median instead of mean - X
# Try calculating metrics over the top 3 players, 5 players, 8 players, and all players (instead of just 8)

# Also see if incorporating availability helps 
# Work on seniority
# Run the same thing for womens


### Evaluate Impact on Overall Model

In [17]:
to_predict_mens_recent = to_predict_mens[(to_predict_mens.Season >= 2009)
        # filter out first four games
        & (to_predict_mens.GameRound >= 1)

        ]

In [ ]:
baseline_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank']
eval_df = validation.run_evaluation_framework(to_predict_mens_recent, pipeline, baseline_features, param_grid, cv_start=2013)

In [ ]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.186917,"(-0.18855628583849895, -0.18527712588509307)",0.187416


In [12]:
cols = [col for col in to_predict_mens_recent.columns[55:] if col not in ['t1_injured_players', 't2_injured_players']]

In [ ]:
def create_matching_tuples(columns):
    # Extract unique suffixes by removing the t1_ and t2_ prefixes
    suffixes = set()
    for col in columns:
        if col.startswith("t1_"):
            suffixes.add(col[3:])
        elif col.startswith("t2_"):
            suffixes.add(col[3:])
    
    # Create matching tuples by prepending the prefixes back
    matching_tuples = [('t1_' + suffix, 't2_' + suffix) for suffix in sorted(suffixes)]
    return matching_tuples

def create_matching_tuples(columns):
    columns_set = set(columns)
    matching_tuples = []
    
    for col in columns:
        if col.startswith("t1_"):
            suffix = col[3:]  # Remove the "t1_" prefix
            parts = suffix.split('_')
            # Expect at least three parts: amount, metric, stat (which may have underscores)
            if len(parts) < 3:
                amount, metric, stat = np.nan, np.nan, np.nan
            else:
                # amount is the first part, metric is the second, stat is everything after
                amount = parts[0]
                metric = parts[1]
                stat = "_".join(parts[2:])  # join remaining parts in case stat contains underscores
            t2_col = "t2_" + suffix
            if t2_col in columns_set:
                matching_tuples.append((col, t2_col, amount, metric, stat))
            else:
                # Optionally, if t2 column is missing, you can choose to append a tuple with np.nan values
                matching_tuples.append((col, np.nan, amount, metric, stat))
    
    return matching_tuples

feature_metadata = create_matching_tuples(cols)
paired_features = [(x[0], x[1]) for x in feature_metadata]


In [26]:
import tqdm

baseline_features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank']

results = []

for pair in tqdm.tqdm(paired_features):
    new_features = list(pair) + baseline_features
    eval_df = validation.run_evaluation_framework(to_predict_mens_recent, pipeline, new_features, param_grid, cv_start=2013)
    result = eval_df["rolling_season_cv"].loc[0]
    results.append(result)


100%|██████████| 315/315 [38:29<00:00,  7.33s/it]


In [29]:
result_analysis = pd.DataFrame(feature_metadata, columns = ["feature1", "feature2", "top_n", "metric", "stat"])
result_analysis["results"] = results


In [42]:
result_analysis.sort_values(by="results")

,feature1,feature2,top_n,metric,stat,results
25,t1_top8_BPM_weighted_mean,t2_top8_BPM_weighted_mean,top8,BPM,weighted_mean,0.180556
128,t1_top5_BPM_mean,t2_top5_BPM_mean,top5,BPM,mean,0.183316
24,t1_top8_BPM_mean,t2_top8_BPM_mean,top8,BPM,mean,0.183503
129,t1_top5_BPM_weighted_mean,t2_top5_BPM_weighted_mean,top5,BPM,weighted_mean,0.183865
133,t1_top5_BPM_median,t2_top5_BPM_median,top5,BPM,median,0.184310
...,...,...,...,...,...,...
46,t1_top8_EFG_weighted_mean,t2_top8_EFG_weighted_mean,top8,EFG,weighted_mean,0.188344
266,t1_top3_OR_max,t2_top3_OR_max,top3,OR,max,0.188395
270,t1_top3_OR_cv,t2_top3_OR_cv,top3,OR,cv,0.188404
271,t1_top3_OR_gini,t2_top3_OR_gini,top3,OR,gini,0.188414


In [39]:
result_analysis.loc[result_analysis.groupby("metric")["results"].idxmin()].sort_values(by="results").feature1.unique()

array(['t1_top8_BPM_weighted_mean', 't1_top8_TO_stdev',
       't1_top5_PRPG!_median', 't1_top3_DR_median', 't1_top5_STL_cv',
       't1_top3_Min%_median', 't1_top8_AST_mean', 't1_top8_TS_gini',
       't1_top8_FTR_mean', 't1_top3_BLK_median',
       't1_injured_players_value', 't1_top8_EFG_gini',
       't1_top5_Games_median', 't1_top3_USG_gini', 't1_top8_OR_mean',
       't1_top5_ORTG_median', 't1_n_injured_players'], dtype=object)

In [20]:
# LATEST
baseline_features = [
    't1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean',
    't1_top8_TO_stdev', 't2_top8_TO_stdev',
    't1_top5_PRPG!_median', 't2_top5_PRPG!_median',
    't1_top3_DR_median', 't2_top3_DR_median',
    't1_top5_STL_cv', 't2_top5_STL_cv',
    't1_top3_Min%_median', 't2_top3_Min%_median',
    't1_top8_TS_gini', 't2_top8_TS_gini',
    't1_top3_USG_gini', 't2_top3_USG_gini',
    't1_OrdinalRank', 't2_OrdinalRank',
    't1_adj_margin', 't2_adj_margin', 
    ]

eval_df = validation.run_evaluation_framework(to_predict_mens_recent, pipeline, baseline_features, param_grid, cv_start=2013)

eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.1},-0.17509,"(-0.176825368366465, -0.17335379961186892)",0.173456


In [ ]:
# 0.180609
# 0.178939
# 0.17739
# 0.176409
# 0.175952
# 0.175767
# 0.174147
# 0.173239
# 0.173117
# 0.172781

In [288]:
og_features = ['t1_adj_margin', 't2_adj_margin', 
    't1_final_rank', 't2_final_rank', 
    't1_OrdinalRank', 't2_OrdinalRank',]

In [289]:
train = to_predict_mens_recent[to_predict_mens_recent.Season < 2024]
test = to_predict_mens_recent[to_predict_mens_recent.Season == 2024]

In [290]:
# Define the classifier and parameter grid
model = LogisticRegression(C=0.1)

In [291]:
model.fit(train[baseline_features], train["Outcome"])

LogisticRegression(C=0.1)

In [292]:
preds = model.predict_proba(test[baseline_features])[:,1]
test["pred"] = preds

/var/folders/h5/f91pbgmj0rj6v0ls3y8zc5l40000gn/T/ipykernel_38700/3789493722.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test["pred"] = preds


In [281]:
test[["t1_TeamName","t2_TeamName", "pred", "Outcome", "GameRound"]].head(50)

,t1_TeamName,t2_TeamName,pred,Outcome,GameRound
1308,Arizona,Long Beach St,0.946914,1,1
1309,Creighton,Akron,0.876735,1,1
1310,Dayton,Nevada,0.517733,1,1
1311,Duquesne,BYU,0.268017,1,1
1312,Gonzaga,McNeese St,0.763205,1,1
1313,Illinois,Morehead St,0.808279,1,1
1314,Iowa St,S Dakota St,0.866040,1,1
1315,Kansas,Samford,0.812878,1,1
1316,Michigan St,Mississippi St,0.610384,1,1
1317,NC State,Texas Tech,0.373175,1,1


In [ ]:
test[["t1_TeamName","t2_TeamName", "pred", "Outcome", "GameRound"]].tail(50)

,t1_TeamName,t2_TeamName,pred,Outcome,GameRound
1308,Arizona,Long Beach St,0.972117,1,1
1309,Creighton,Akron,0.875513,1,1
1310,Dayton,Nevada,0.519298,1,1
1311,Duquesne,BYU,0.171792,1,1
1312,Gonzaga,McNeese St,0.872135,1,1
1313,Illinois,Morehead St,0.963656,1,1
1314,Iowa St,S Dakota St,0.880621,1,1
1315,Kansas,Samford,0.749063,1,1
1316,Michigan St,Mississippi St,0.537957,1,1
1317,NC State,Texas Tech,0.569893,1,1


In [ ]:
# 0.183256

In [ ]:
# 0.186888
# 0.183598
# 0.182179
# 0.180219
# 0.178096
# 0.176353
# 0.176224
# 0.176138
# 0.176077
# 0.175832
# 0.175375
# 0.174467

In [220]:
result_analysis.sort_values(by="results").head(50)

,feature1,feature2,results
19,t1_BPM_mean,t2_BPM_mean,0.183598
60,t1_TO_stdev,t2_TO_stdev,0.184625
62,t1_TO_gini,t2_TO_gini,0.185160
61,t1_TO_cv,t2_TO_cv,0.185217
58,t1_TO_max,t2_TO_max,0.185415
14,t1_PRPG!_mean,t2_PRPG!_mean,0.185593
54,t1_AST_mean,t2_AST_mean,0.185990
42,t1_TS_gini,t2_TS_gini,0.186012
12,t1_Min%_gini,t2_Min%_gini,0.186147
41,t1_TS_cv,t2_TS_cv,0.186224


In [ ]:
# features = ['t1_adj_margin', 't2_adj_margin', 't1_final_rank', 't2_final_rank', 't1_OrdinalRank', 't2_OrdinalRank',
#             't1_injured_players_value',	't2_injured_players_value',
#             't1_Min%_stdev', 't2_Min%_stdev',
#             't1_PRPG!_max',	't2_PRPG!_max',
#             't1_USG_stdev','t2_USG_stdev',
#             't1_TS_stdev', 't2_TS_stdev',
#             't1_OR_mean', 't2_OR_mean',
#             't1_AST_mean', 't2_AST_mean',
#             't1_BLK_cv', 't2_BLK_cv',
#             't1_STL_cv', 't2_STL_cv',
#             't1_FTR_mean', 't2_FTR_mean'	
#             ]

# features = [
    # 't1_adj_margin', 't2_adj_margin', 
    # 't1_final_rank', 't2_final_rank', 
    # 't1_OrdinalRank', 't2_OrdinalRank',
            # 't1_BPM_mean',	't2_BPM_mean', 't1_TO_stdev', 't2_TO_stdev', 't1_AST_mean',	't2_AST_mean',
            # 't1_TS_gini', 't2_TS_gini', 
            # 't1_BLK_cv', 't2_BLK_cv', 
            # 't1_health_score', 't2_health_score',
            # 't1_Min%_gini',	't2_Min%_gini',	
            # 't1_EFG_cv', 't2_EFG_cv',
            # 't1_FTR_mean', 't2_FTR_mean'
            # ]

\
eval_df

[[ 0.15996993 -0.15996993  0.20803112 -0.20803112 -0.15068604  0.15068604
   0.0506448  -0.0506448  -0.02207514  0.02207514  0.11662714 -0.11662714
  -0.03582874  0.03582874 -0.01844512  0.01844512  0.04053221 -0.04053221
  -0.03363456  0.03363456 -0.03302173  0.03302173 -0.04269475  0.04269475
   0.02217189 -0.02217189]]
[[ 0.17480693 -0.17480693  0.23317954 -0.23317954 -0.19185949  0.19185949
   0.01225051 -0.01225051 -0.00868236  0.00868236  0.0957119  -0.0957119
  -0.02706569  0.02706569 -0.00360075  0.00360075  0.05822528 -0.05822528
  -0.03999617  0.03999617  0.01499206 -0.01499206 -0.03092588  0.03092588
   0.00025716 -0.00025716]]
[[ 0.17250156 -0.17250156  0.24607687 -0.24607687 -0.21494277  0.21494277
  -0.018696    0.018696   -0.01252743  0.01252743  0.08942567 -0.08942567
  -0.0185512   0.0185512   0.00044711 -0.00044711  0.07899645 -0.07899645
  -0.05299117  0.05299117  0.00482578 -0.00482578 -0.02768647  0.02768647
  -0.01809772  0.01809772]]
[[ 0.19678547 -0.19678547  0.

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.01},-0.185245,"(-0.18666561891754252, -0.18382461356522317)",0.184542


In [ ]:
0.185962



In [ ]:
# 0.186888
# 0.186463
# 0.186291
# 0.186221
# 0.185435
# 0.185274
# 0.185001
# 0.184776
# 0.184592
# 0.184425
# 0.183836


In [125]:
result_analysis = pd.DataFrame(paired_features, columns = ["feature1", "feature2"])
result_analysis["results"] = results

In [171]:
to_predict_mens_recent[result_analysis[result_analysis.results < 0.186888].feature1.tolist()].corr()

,t1_injured_players_value,t1_health_score,t1_Min%_stdev,t1_Min%_cv,t1_Min%_gini,t1_PRPG!_max,t1_PRPG!_mean,t1_PRPG!_stdev,t1_BPM_mean,t1_USG_stdev,...,t1_TS_gini,t1_OR_mean,t1_AST_mean,t1_TO_max,t1_TO_stdev,t1_TO_cv,t1_TO_gini,t1_BLK_cv,t1_STL_cv,t1_FTR_mean
t1_injured_players_value,1.000000,-0.835971,-0.012084,-0.000366,-0.000340,-0.008293,0.048705,-0.021869,0.028900,-0.046716,...,-0.028886,-0.035323,0.002031,0.031795,0.023604,0.034108,0.025152,0.006312,0.047083,-0.035458
t1_health_score,-0.835971,1.000000,-0.071303,-0.068333,-0.061367,0.014897,-0.009196,0.004332,0.008817,0.022725,...,0.010725,0.048822,0.026600,-0.019607,-0.014915,-0.028992,-0.026358,-0.016306,-0.035763,0.010579
t1_Min%_stdev,-0.012084,-0.071303,1.000000,0.986287,0.974257,0.361639,0.180439,0.553214,-0.141290,0.165801,...,0.184296,-0.031167,-0.313852,0.085439,0.140897,0.176387,0.171926,-0.078789,0.029172,-0.054121
t1_Min%_cv,-0.000366,-0.068333,0.986287,1.000000,0.992710,0.349970,0.144535,0.547812,-0.146362,0.166524,...,0.188757,-0.021382,-0.291758,0.104231,0.152160,0.177853,0.172034,-0.067556,0.031379,-0.054210
t1_Min%_gini,-0.000340,-0.061367,0.974257,0.992710,1.000000,0.352223,0.131393,0.549093,-0.155302,0.176063,...,0.182137,-0.022592,-0.285712,0.100483,0.146984,0.170292,0.165405,-0.062120,0.033375,-0.059124
t1_PRPG!_max,-0.008293,0.014897,0.361639,0.349970,0.352223,1.000000,0.679721,0.823247,0.429962,0.235670,...,0.111695,0.112253,-0.067530,-0.042257,0.062982,0.159973,0.151790,-0.054878,-0.019833,-0.119249
t1_PRPG!_mean,0.048705,-0.009196,0.180439,0.144535,0.131393,0.679721,1.000000,0.401097,0.772067,0.025562,...,-0.084592,0.238694,0.050356,-0.286220,-0.152952,0.025987,0.020086,-0.110664,-0.090274,-0.166675
t1_PRPG!_stdev,-0.021869,0.004332,0.553214,0.547812,0.549093,0.823247,0.401097,1.000000,0.126659,0.320670,...,0.322537,0.036045,-0.154506,0.207046,0.269261,0.285172,0.272093,-0.047689,0.008463,-0.049687
t1_BPM_mean,0.028900,0.008817,-0.141290,-0.146362,-0.155302,0.429962,0.772067,0.126659,1.000000,-0.081152,...,-0.117225,0.363717,0.189416,-0.239776,-0.177261,-0.078249,-0.078441,-0.050866,-0.132019,-0.047732
t1_USG_stdev,-0.046716,0.022725,0.165801,0.166524,0.176063,0.235670,0.025562,0.320670,-0.081152,1.000000,...,0.061640,-0.131028,-0.100542,0.005018,0.067681,0.116774,0.114664,0.052443,0.056622,-0.076170


In [169]:
result_analysis[result_analysis.results < 0.186888].feature1.tolist()

['t1_injured_players_value',
 't1_health_score',
 't1_Min%_stdev',
 't1_Min%_cv',
 't1_Min%_gini',
 't1_PRPG!_max',
 't1_PRPG!_mean',
 't1_PRPG!_stdev',
 't1_BPM_mean',
 't1_USG_stdev',
 't1_USG_gini',
 't1_EFG_stdev',
 't1_EFG_cv',
 't1_EFG_gini',
 't1_TS_stdev',
 't1_TS_cv',
 't1_TS_gini',
 't1_OR_mean',
 't1_AST_mean',
 't1_TO_max',
 't1_TO_stdev',
 't1_TO_cv',
 't1_TO_gini',
 't1_BLK_cv',
 't1_STL_cv',
 't1_FTR_mean']

In [ ]:
           # 't1_BPM_mean',	't2_BPM_mean', 't1_TO_stdev', 't2_TO_stdev', 't1_AST_mean',	't2_AST_mean',
            # 't1_TS_gini', 't2_TS_gini', 
            # 't1_BLK_cv', 't2_BLK_cv', 
            # 't1_health_score', 't2_health_score',
            # 't1_Min%_gini',	't2_Min%_gini',	
            # 't1_EFG_cv', 't2_EFG_cv',
            # 't1_FTR_mean', 't2_FTR_mean'

In [ ]:
new = result_analysis[result_analysis.results < 0.186888].feature1.tolist() + result_analysis[result_analysis.results < 0.182191].feature2.tolist()

In [ ]:
t1_injured_players_value	t2_injured_players_value

In [102]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.186142,"(-0.18770587104089648, -0.1845780448838703)",0.186463


In [ ]:
# eff margin: 0.190491
# final rank: 0.189166
# ordinal rank 0.206132
# injury 0.249987
# combined: 0.186615


Not as big of an impact of downstream rounds, but still does make a difference 

Baseline Stats Model:-0.1865, 0.186888

Baseline Stats using all data _, 0.186995

Best New Injury Features: -0.186142, 0.186463

Best New Injury + Other derived player stats: -0.185488, 0.185838

Interesting, there's some evidence here that using only more recent data might be better

---

Overall, I care most about the rolling season cv, and the injury features seem to improve the model on all apples to apples comparions 

ALSO - it seems that the other player stats that aren't related to injury are useful too to the point where even if we don't use the injury data we still see some nice improvements

This is great because while the injury data has some leakage the other stats shouldn't

---

Individual Impacts
Injury Stats: 0.0004250000000000087
Depth Stats: 0.0007209999999999994
Guard Stats: 0.0000170000


In [124]:
num = .186888 - 0.186871
print(f"{num:.10f}") 

0.0000170000
